# Занятие 3. Линейная классификация

На прошлом занятии ответом было число, и линейная модель предсказывала его
напрямую. Сегодня ответ — метка класса. Модель остается той же линейной
функцией признаков, меняется только то, как ее выход превращается в ответ,
и функция потерь. Вся вероятностная логика прошлого занятия повторится:
предположение о том, как устроен ответ, дает функцию потерь, а штраф —
предположение о весах.

Датасет тот же, вино. Сначала два признака и два класса, чтобы все можно было
нарисовать, потом все признаки и все три сорта. Три модели — регрессия
на метках, логистическая регрессия и метод опорных векторов — будут
сравниваться на одних и тех же данных, одними метриками и на одной картинке.

**Как работать с ноутбуком.** Ячейки идут по порядку, запускать сверху вниз.
Три ячейки с многоточиями — функции, которые пишем вместе, за каждой идет
проверка. Остальные ячейки готовы: запускайте, смотрите на числа и меняйте
параметры там, где предложено.

## 0. Подготовка

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, log_loss, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=ConvergenceWarning)

plt.rcParams.update({
    "figure.figsize": (8, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BLUE, BLACK, OCHRE, GREY = "#0072CE", "#0F1418", "#C98A3C", "#9CA3AF"
SEED = 42

## 1. Задача: сорт 1 или нет

Начнем с двух классов: вино сорта 1 против всех остальных. Целевая переменная
равна единице для сорта 1 и нулю для сортов 0 и 2. Чтобы все можно было
нарисовать, берем два признака: флавоноиды и интенсивность цвета.

Делим данные со `stratify`: в обеих частях доля сорта 1 должна быть одинаковой,
иначе тестовая выборка окажется случайно перекошенной.

In [ ]:
wine = load_wine(as_frame=True)
frame = wine.frame

y_binary = (frame["target"] == 1).astype(int).to_numpy()
pair_names = ["flavanoids", "color_intensity"]
X_pair = frame[pair_names].to_numpy()

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_pair, y_binary, test_size=0.3, random_state=SEED, stratify=y_binary
)
scaler_pair = StandardScaler().fit(X_train_raw)
X_train = scaler_pair.transform(X_train_raw)
X_test = scaler_pair.transform(X_test_raw)

print("train:", X_train.shape, " test:", X_test.shape)
print(f"доля сорта 1: в обучении {y_train.mean():.2f}, в тесте {y_test.mean():.2f}")

In [ ]:
def plot_points(ax, X, y, alpha=0.8):
    ax.scatter(X[y == 0, 0], X[y == 0, 1], s=28, color=GREY, alpha=alpha, label="другие сорта")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], s=28, color=BLUE, alpha=alpha, label="сорт 1")
    ax.set_xlabel("flavanoids, стандартизованные")
    ax.set_ylabel("color_intensity, стандартизованные")


fig, ax = plt.subplots(figsize=(6.5, 5))
plot_points(ax, X_train, y_train)
ax.set_title("Обучающая выборка")
ax.legend()
plt.show()

Классы почти разделяются прямой, но не совсем: у сорта 1 низкая интенсивность
цвета, а флавоноиды по нему ничего не гарантируют. Идеально провести границу
не получится, и это нормально.

## 2. Почему не линейная регрессия

Метки 0 и 1 — числа. Почему бы не обучить на них линейную регрессию
с прошлого занятия и не проводить границу там, где предсказание равно 0.5?
Попробуем на наших винах.

In [ ]:
regression_model = LinearRegression().fit(X_train, y_train)
w_reg, b_reg = regression_model.coef_, regression_model.intercept_ - 0.5    # граница: предсказание = 0.5


def boundary_line(ax, w, b, color=BLACK, label="граница", band=False):
    xs = np.linspace(X_train[:, 0].min() - 0.3, X_train[:, 0].max() + 0.3, 100)
    # на прямой w0 x + w1 y + b = 0, отсюда y
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=color, lw=2, label=label)
    if band:
        for shift in (-1, 1):
            ax.plot(xs, -(w[0] * xs + b - shift) / w[1], color=color, lw=1, ls="--")


errors_reg = ((X_train @ w_reg + b_reg >= 0).astype(int) != y_train)

fig, ax = plt.subplots(figsize=(6.5, 5))
plot_points(ax, X_train, y_train, alpha=0.6)
boundary_line(ax, w_reg, b_reg, label="предсказание регрессии = 0.5")
ax.scatter(X_train[errors_reg, 0], X_train[errors_reg, 1], s=90, facecolor="none", edgecolor=OCHRE, lw=1.8, label="ошибки")
ax.set_title(f"Линейная регрессия на метках: ошибок {errors_reg.sum()} из {len(y_train)}")
ax.legend(loc="upper right")
plt.show()

Работает: прямая проходит между классами, ошибок шестнадцать. Теперь
добавим в обучающую выборку пятнадцать вин сорта 1, которые лежат далеко
от границы на своей стороне: очень низкая интенсивность цвета. Для
классификации это самые легкие объекты, они и так на правильной стороне
с большим запасом. Посмотрим, что с ними сделает регрессия.

In [ ]:
rng_far = np.random.default_rng(SEED)
far_wines = rng_far.normal([-1.5, -6.0], 0.4, (15, 2))
X_with_far = np.vstack([X_train, far_wines])
y_with_far = np.r_[y_train, np.ones(15, dtype=int)]

regression_far = LinearRegression().fit(X_with_far, y_with_far)
w_far, b_far = regression_far.coef_, regression_far.intercept_ - 0.5
errors_far = ((X_train @ w_far + b_far >= 0).astype(int) != y_train)

fig, ax = plt.subplots(figsize=(6.5, 7))
plot_points(ax, X_train, y_train, alpha=0.6)
ax.scatter(far_wines[:, 0], far_wines[:, 1], s=28, color=BLUE, marker="s", label="добавленные вина сорта 1")
boundary_line(ax, w_reg, b_reg, color=GREY, label="граница до добавления")
boundary_line(ax, w_far, b_far, label="граница после добавления")
ax.scatter(X_train[errors_far, 0], X_train[errors_far, 1], s=90, facecolor="none", edgecolor=OCHRE, lw=1.8, label="ошибки на исходных винах")
ax.set_title(f"Ошибок на исходных 124 винах: было {errors_reg.sum()}, стало {errors_far.sum()}")
ax.legend(loc="upper right")
plt.show()

print("веса до:   ", np.round(w_reg, 3), " после:", np.round(w_far, 3))

Добавили объекты, которые классифицированы верно с огромным запасом, —
и граница ушла, а ошибок на исходных винах стало больше. Регрессия
штрафует далекие вина за то, что предсказание для них не равно ровно
единице, а сильно больше, и разворачивает прямую, чтобы уменьшить этот
квадрат. Для классификации это бессмысленно: уверенно правильный ответ
не должен ничего стоить. Нужна функция потерь, которая правильные ответы
не трогает, и весь дальнейший разговор — о том, какая именно.

На исходных винах линейная регрессия на метках все же работает, и она станет
первой моделью в сравнении.

## 3. Линейный классификатор и отступ

Линейная модель, как и раньше, считает $f(x) = \langle w, x \rangle + b$. Ответом
классификатора служит знак этого числа. Для двух классов удобно кодировать
метки как $y \in \{-1, +1\}$:

$$a(x) = \mathrm{sign}\,\big(\langle w, x \rangle + b\big)$$

Граница между классами — множество точек, где $\langle w, x \rangle + b = 0$.
На плоскости это прямая, в пространстве признаков — гиперплоскость.
Вектор $w$ перпендикулярен ей.

Главная величина сегодняшнего занятия — **отступ**:

$$M = y \cdot \big(\langle w, x \rangle + b\big)$$

Знак отступа говорит, верно ли классифицирован объект, модуль — насколько
далеко он от границы. Отступ положительный: объект по правильную сторону.
Отрицательный: ошибка. Чем больше отступ, тем увереннее ответ.

Любой линейный классификатор — это пара $(w, b)$, поэтому все модели сегодня
проходят через две общие функции: `evaluate_model` считает одни и те же метрики
на одном и том же тесте и добавляет строку в таблицу `results`,
`show_model` рисует границу, ошибки и отступы на обучающей выборке.

In [ ]:
results = {}


def margins_of(X, y, w, b):
    return (2 * y - 1) * (X @ w + b)                  # метки -1 и +1


def evaluate_model(name, w, b):
    predicted = (X_test @ w + b >= 0).astype(int)
    results[name] = {
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted),
        "recall": recall_score(y_test, predicted),
        "ошибок на обучении": int((margins_of(X_train, y_train, w, b) < 0).sum()),
    }
    return pd.DataFrame(results).T.round(3)


def show_model(name, w, b, band=False, support=None):
    m = margins_of(X_train, y_train, w, b)
    wrong = m < 0

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
    plot_points(axes[0], X_train, y_train, alpha=0.5)
    boundary_line(axes[0], w, b, band=band)
    axes[0].scatter(X_train[wrong, 0], X_train[wrong, 1], s=90, facecolor="none", edgecolor=OCHRE, lw=1.8, label="ошибки")
    if support is not None:
        axes[0].scatter(X_train[support, 0], X_train[support, 1], s=150, facecolor="none", edgecolor=BLACK, lw=1.0, label="опорные векторы")
    axes[0].set_title(f"{name}: ошибок на обучении {wrong.sum()}")
    axes[0].legend(loc="upper right")

    bins = np.linspace(-4, 16, 41)
    axes[1].hist(m[m >= 0], bins=bins, color=BLUE, label="отступ > 0: верно")
    axes[1].hist(m[m < 0], bins=bins, color=OCHRE, label="отступ < 0: ошибка")
    axes[1].axvline(0, color=BLACK, lw=1)
    axes[1].set_xlabel("отступ M")
    axes[1].set_ylabel("сколько вин")
    axes[1].set_title("Отступы обучающих вин")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

Первая модель — линейная регрессия на метках из второго раздела: $w$ — ее
веса, $b$ — свободный член минус 0.5.

In [ ]:
show_model("линейная регрессия на метках", w_reg, b_reg)
evaluate_model("линейная регрессия на метках", w_reg, b_reg)

In [ ]:
# кто именно на чужой стороне: маленький отрицательный отступ — вино почти на границе,
# большой по модулю — уверенная ошибка
m_reg = margins_of(X_train, y_train, w_reg, b_reg)
wrong_wines = pd.DataFrame(X_train_raw[m_reg < 0], columns=pair_names)
wrong_wines["сорт 1"] = y_train[m_reg < 0]
wrong_wines["отступ"] = m_reg[m_reg < 0].round(2)
wrong_wines.sort_values("отступ")

Отступ можно считать для любой прямой. Дальше вопрос, какую прямую выбрать,
то есть какую функцию потерь минимизировать.

## 4. Функции потерь через отступ

Хочется минимизировать число ошибок, то есть долю объектов с $M < 0$.
Функция потерь для одного объекта — индикатор $[M < 0]$. Она ступенчатая:
градиент всюду ноль, кроме точки $M = 0$, где его нет. Спуску не за что
зацепиться.

Все линейные классификаторы делают одно и то же: заменяют ступеньку гладкой
или хотя бы непрерывной функцией, которая лежит выше нее. Три главных
кандидата:

- **квадратичная** $(1 - M)^2$ — линейная регрессия на метках $\pm 1$, ее мы уже видели;
- **логистическая** $\log_2(1 + e^{-M})$ — логистическая регрессия;
- **кусочно-линейная** $\max(0, 1 - M)$ — метод опорных векторов.

Все три в точке $M = 0$ равны единице, как и ступенька, поэтому на графике
они касаются ее в одном месте. У логистической для этого взят логарифм
по основанию 2; на минимум это не влияет, множитель перед функцией потерь
положение минимума не сдвигает.

У квадратичной есть та самая беда из второго раздела: она растет и при
больших положительных отступах, то есть наказывает уверенно правильные ответы.

In [ ]:
margin_grid = np.linspace(-3, 4, 400)
losses = {
    "[M < 0]: доля ошибок": ((margin_grid < 0).astype(float), BLACK),
    "(1 - M)^2: квадратичная, линейная регрессия на метках": ((1 - margin_grid) ** 2, GREY),
    "log2(1 + exp(-M)): логистическая": (np.log1p(np.exp(-margin_grid)) / np.log(2), BLUE),
    "max(0, 1 - M): hinge, SVM": (np.maximum(0, 1 - margin_grid), OCHRE),
}

fig, ax = plt.subplots(figsize=(8, 4.5))
for name, (values, color) in losses.items():
    ax.plot(margin_grid, values, color=color, lw=2.2, label=name)
ax.set_ylim(0, 4.5)
ax.set_xlabel("отступ M")
ax.set_ylabel("потери на одном объекте")
ax.set_title("Чем заменить ступеньку")
ax.legend()
plt.show()

## 5. Логистическая регрессия

Повторяем ход прошлого занятия. Там мы предположили, что ответ — это
$\langle w, x \rangle$ плюс нормальный шум, и из правдоподобия получили квадраты.
Теперь ответ — ноль или единица, и ему подходит распределение Бернулли:
объект относится к классу 1 с вероятностью $p$ и к классу 0 с вероятностью $1 - p$.

Вероятность должна зависеть от признаков и лежать в отрезке от нуля
до единицы, а $\langle w, x \rangle + b$ пробегает всю числовую прямую.
Сжимает ее в отрезок **сигмоида**:

$$\sigma(z) = \frac{1}{1 + e^{-z}}, \qquad
p(y = 1 \mid x) = \sigma\big(\langle w, x \rangle + b\big)$$

Правдоподобие выборки — произведение по объектам, у каждого либо $p_i$, либо
$1 - p_i$. Логарифм со знаком минус, усредненный по объектам, называется
**log loss**, он же кросс-энтропия:

$$L = -\frac{1}{n}\sum_{i=1}^{n}\Big(y_i \log p_i + (1 - y_i)\log(1 - p_i)\Big)$$

Скобка выглядит громоздко, но в ней всегда выживает одно слагаемое: при
$y_i = 1$ остается $\log p_i$, при $y_i = 0$ остается $\log(1 - p_i)$.
То есть это просто логарифм вероятности, которую модель дала правильному
ответу, $p(y_i \mid x_i)$:

$$L = -\frac{1}{n}\sum_{i=1}^{n} \log p(y_i \mid x_i)
= -\frac{1}{n}\sum_{i=1}^{n} \sum_{k \in \{0, 1\}} [y_i = k]\, \log p(y = k \mid x_i)$$

В таком виде формула не зависит от числа классов и в девятом разделе
без изменений станет функцией потерь для трех сортов.

Для меток $\pm 1$ то же самое записывается через отступ как
$\frac{1}{n}\sum \log(1 + e^{-M_i})$: это синяя кривая с прошлого графика.
Максимум правдоподобия — минимум log loss.

Напишем сигмоиду и log loss. У log loss два подводных камня: логарифм нуля
и переполнение экспоненты. Вероятности перед логарифмом прижимайте к отрезку
$[\varepsilon, 1 - \varepsilon]$ через `np.clip`, а в сигмоиде `np.exp(-z)`
при большом отрицательном $z$ переполняется: результат `inf` и `1 / (1 + inf)`
равен нулю, что верно, но numpy выдаст предупреждение. Глушится оно блоком
`with np.errstate(over="ignore"):`.

In [ ]:
# z: массив любой формы -> массив той же формы со значениями в (0, 1)
def sigmoid(z: np.ndarray) -> np.ndarray:
    return ...


# y_true: (n,) из нулей и единиц, proba: (n,) вероятности класса 1 -> число
def logistic_loss(y_true: np.ndarray, proba: np.ndarray, eps: float = 1e-12) -> float:
    return ...

In [ ]:
# --- проверка ---

# 1. сигмоида: ноль в середине, симметрия, края
assert np.isclose(sigmoid(0), 0.5), "sigmoid(0) должна быть 0.5"
assert np.allclose(sigmoid(np.array([-2.0, 2.0])).sum(), 1.0), "sigmoid(-z) + sigmoid(z) = 1"
assert sigmoid(np.array([-1000.0, 1000.0])).tolist() == [0.0, 1.0], "на краях сигмоида должна давать ровно 0 и 1"

# 2. log loss совпадает с sklearn
proba_check = sigmoid(X_train @ w_reg + b_reg)
assert np.isclose(logistic_loss(y_train, proba_check), log_loss(y_train, proba_check)), "log loss не совпал со sklearn"

# 3. уверенная ошибка стоит дорого, идеальный ответ — ноль
assert logistic_loss([1], [1e-12]) > 20, "уверенная ошибка должна давать большие потери"
assert np.isclose(logistic_loss([0, 1], [0.0, 1.0]), 0.0, atol=1e-10), "на идеальном ответе потери около нуля"

print("проверки пройдены")

## 6. Градиент и спуск

Формулы для весов, как в линейной регрессии, здесь нет: уравнение
$\nabla L = 0$ нелинейное. Остается градиентный спуск.

### Вывод градиента

Как на прошлом занятии, $w$ включает свободный член, а $X$ — матрица
признаков со столбцом единиц. Обозначаем $z_i = \langle w, x_i \rangle$,
$p_i = \sigma(z_i)$. Функция потерь из пятого
раздела:

$$L(w) = -\frac{1}{n}\sum_{i=1}^{n}\Big(y_i \log p_i + (1 - y_i)\log(1 - p_i)\Big)$$

Считаем градиент по цепочке: $L$ зависит от $p_i$, $p_i$ от $z_i$, $z_i$
от $w$.

**Шаг 1: производная сигмоиды.** Прямое дифференцирование дроби:

$$\sigma'(z) = \frac{e^{-z}}{(1 + e^{-z})^2}
= \frac{1}{1 + e^{-z}} \cdot \frac{e^{-z}}{1 + e^{-z}}
= \sigma(z)\big(1 - \sigma(z)\big)$$

Второй множитель — это $1 - \sigma(z)$, потому что
$1 - \frac{1}{1 + e^{-z}} = \frac{e^{-z}}{1 + e^{-z}}$.

**Шаг 2: производная потерь одного объекта по $z_i$.** Обозначим
$\ell_i = -\big(y_i \log p_i + (1 - y_i)\log(1 - p_i)\big)$. Сначала по $p_i$:

$$\frac{\partial \ell_i}{\partial p_i} = -\frac{y_i}{p_i} + \frac{1 - y_i}{1 - p_i}$$

Умножаем на $\frac{\partial p_i}{\partial z_i} = p_i(1 - p_i)$ из первого шага:

$$\frac{\partial \ell_i}{\partial z_i}
= \Big(-\frac{y_i}{p_i} + \frac{1 - y_i}{1 - p_i}\Big)\, p_i (1 - p_i)
= -y_i(1 - p_i) + (1 - y_i)\, p_i
= p_i - y_i$$

Знаменатели сократились, и осталась разность между вероятностью и меткой.
Именно из-за этого сокращения log loss и сигмоида так хорошо подходят друг
к другу.

**Шаг 3: по весам.** $z_i = \langle w, x_i \rangle$, поэтому
$\frac{\partial z_i}{\partial w} = x_i$, и для одного объекта

$$\frac{\partial \ell_i}{\partial w} = (p_i - y_i)\, x_i$$

Проверим размерности: $p_i - y_i$ — одно число, $x_i$ — вектор длины $d + 1$,
их произведение — вектор длины $d + 1$. Столько же компонент у $w$,
как и должно быть у градиента по $w$.

**Шаг 4: сумма по объектам.** $L = \frac{1}{n}\sum_i \ell_i$, значит

$$\nabla_w L = \frac{1}{n}\sum_{i=1}^{n} (p_i - y_i)\, x_i$$

Это сумма $n$ векторов длины $d + 1$, каждый со своим числовым
коэффициентом $(p_i - y_i)$. Такая сумма — это произведение транспонированной
матрицы признаков на вектор коэффициентов: $X^\top$ имеет размер
$(d + 1) \times n$, вектор $p - y$ — длину $n$, результат — длину $d + 1$. Строки $X$ —
это $x_i$, $p = \sigma(Xw)$ — вектор всех вероятностей, и

$$\nabla_w L = \frac{1}{n} X^\top (p - y)$$

Сравните с линейной регрессией: там было $\frac{2}{n} X^\top (Xw - y)$.
Разница только в том, что вместо линейного предсказания стоит вероятность,
и нет двойки от производной квадрата.

### Спуск

Спуск начинается с нулевого вектора, шаг $\eta$, после каждой итерации
в историю дописывается log loss.

In [ ]:
# X: (n, d + 1) — признаки со столбцом единиц, y: (n,) из нулей и единиц, w: (d + 1,)
# возвращает градиент log loss по w: (d + 1,)
def logistic_gradient(X: np.ndarray, y: np.ndarray, w: np.ndarray) -> np.ndarray:
    return ...


# возвращает w: (d + 1,) после n_iter шагов и history — список из n_iter значений log loss
def fit_logistic(X: np.ndarray, y: np.ndarray, lr: float, n_iter: int) -> tuple[np.ndarray, list[float]]:
    w: np.ndarray = ...
    history: list[float] = ...
    for _ in range(n_iter):
        ...
    return w, history

In [ ]:
# --- проверка ---
X_ones = np.column_stack([np.ones(len(X_train)), X_train])

# 1. градиент в нуле: вероятности 0.5, значит (p - y) = 0.5 - y
expected_zero = X_ones.T @ (0.5 - y_train) / len(y_train)
assert np.allclose(logistic_gradient(X_ones, y_train, np.zeros(3)), expected_zero), "градиент в нулевой точке посчитан неверно"

# 2. история нужной длины и потери не растут
w_gd, history_gd = fit_logistic(X_ones, y_train, lr=0.5, n_iter=3000)
assert len(history_gd) == 3000, "длина history должна равняться n_iter"
assert all(later <= earlier + 1e-9 for earlier, later in zip(history_gd, history_gd[1:])), "при таком шаге потери не должны расти"

# 3. спуск приходит туда же, куда sklearn без штрафа
logistic_model = LogisticRegression(C=np.inf, max_iter=10000).fit(X_train, y_train)
w_sklearn = np.r_[logistic_model.intercept_, logistic_model.coef_[0]]
assert np.allclose(w_gd, w_sklearn, atol=1e-2), (
    f"спуск должен сойтись к весам sklearn: {np.round(w_gd, 3)} против {np.round(w_sklearn, 3)}"
)

print("проверки пройдены")
print(f"после 3000 шагов: w = {np.round(w_gd, 3)}, log loss {history_gd[-1]:.4f}")

Веса найдены. Смотрим на логистическую регрессию так же, как на регрессию
на метках: та же картинка, те же метрики, та же таблица.

In [ ]:
w_log, b_log = w_gd[1:], w_gd[0]

show_model("логистическая регрессия", w_log, b_log)
evaluate_model("логистическая регрессия", w_log, b_log)

Разница видна сразу. Граница логистической регрессии почти горизонтальна:
сорт 1 отличает интенсивность цвета, флавоноиды почти ни при чем. Граница
линейной регрессии на метках наклонена и ошибается на четыре вина больше. Это тот же эффект,
что во втором разделе: вина других сортов с очень высокой интенсивностью
цвета классифицированы верно, но регрессия штрафует их за предсказание,
далекое от нуля, и разворачивает прямую к ним.

Гистограммы отступов сравнивать по масштабу нельзя: у регрессии отступ —
это предсказание минус 0.5, оно всегда в пределах единицы, а у логистической
регрессии это логарифм отношения шансов, и он не ограничен. Сравнивать
можно форму: у логистической регрессии есть длинный правый хвост уверенно
правильных ответов, которых ничто не сдерживает.

### Где сигмоида

В пятом разделе вероятность была сигмоидой от $\langle w, x \rangle + b$,
а на картинке выше — прямая. Противоречия нет: по обеим осям здесь признаки,
а выход модели, вероятность, на картинке вообще не показан. Прямая — это
только линия уровня $p = 0.5$, то есть $\langle w, x \rangle + b = 0$.

Покажем вероятность цветом, и сигмоида найдется: она поднимается над
плоскостью поперек границы. Рядом обе прямые на одних осях.

In [ ]:
grid_x, grid_y = np.meshgrid(
    np.linspace(X_train[:, 0].min() - 0.3, X_train[:, 0].max() + 0.3, 200),
    np.linspace(X_train[:, 1].min() - 0.3, X_train[:, 1].max() + 0.3, 200),
)
grid_points = np.column_stack([grid_x.ravel(), grid_y.ravel()])
proba_grid = sigmoid(grid_points @ w_log + b_log).reshape(grid_x.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
heat = axes[0].pcolormesh(grid_x, grid_y, proba_grid, cmap="RdBu_r", vmin=0, vmax=1, shading="auto", alpha=0.85)
fig.colorbar(heat, ax=axes[0], label="вероятность сорта 1")
plot_points(axes[0], X_train, y_train, alpha=0.8)
boundary_line(axes[0], w_log, b_log, color=BLACK, label="p = 0.5")
axes[0].set_title("Выход логистической регрессии над плоскостью")
axes[0].legend(loc="upper right")

plot_points(axes[1], X_train, y_train, alpha=0.35)
boundary_line(axes[1], w_reg, b_reg, color=GREY, label="линейная регрессия на метках")
boundary_line(axes[1], w_log, b_log, color=BLUE, label="логистическая регрессия")
axes[1].set_title("Две границы на одной плоскости")
axes[1].legend(loc="upper right")
plt.tight_layout()
plt.show()

print(f"линейная регрессия на метках:      w = {np.round(w_reg, 3)}, b = {b_reg:.3f}")
print(f"логистическая регрессия:  w = {np.round(w_log, 3)}, b = {b_log:.3f}")

Слева цвет меняется от синего к красному поперек границы плавно: это
и есть сигмоида, только смотрим на нее сверху. Вдоль границы цвет
не меняется, потому что линейная модель зависит от признаков только через
одно число $\langle w, x \rangle + b$. Справа видно, что граница регрессии
на метках наклонена, а логистической почти горизонтальна: во второй вес
флавоноидов близок к нулю.

In [ ]:
# что модель думает про крайние вина: самое уверенное «сорт 1», самое уверенное «нет» и самое сомнительное
proba_train = sigmoid(X_train @ w_log + b_log)
for label, idx in [("уверенно сорт 1", proba_train.argmax()),
                   ("уверенно не сорт 1", proba_train.argmin()),
                   ("на границе", np.abs(proba_train - 0.5).argmin())]:
    print(f"{label:20s} p = {proba_train[idx]:.3f}   flavanoids {X_train_raw[idx, 0]:.2f}, "
          f"color_intensity {X_train_raw[idx, 1]:.2f}, на самом деле сорт 1: {bool(y_train[idx])}")
print(f"\nlog loss на обучении {logistic_loss(y_train, proba_train):.3f}, на тесте "
      f"{logistic_loss(y_test, sigmoid(X_test @ w_log + b_log)):.3f}")

In [ ]:
# скорость сходимости при разных шагах: попробуйте добавить lr побольше и посмотреть, когда разойдется
fig, ax = plt.subplots(figsize=(8, 4.2))
for lr, color in [(0.05, GREY), (0.2, OCHRE), (0.5, BLUE), (1.5, BLACK)]:
    _, history = fit_logistic(X_ones, y_train, lr=lr, n_iter=400)
    ax.plot(np.arange(1, 401), history, color=color, lw=1.8, label=f"шаг {lr}")
ax.set_xlabel("итерация")
ax.set_ylabel("log loss")
ax.set_title("Спуск при разных шагах")
ax.legend()
plt.show()

Кривые расходятся не так драматично, как в линейной регрессии: у log loss
градиент ограничен, вероятность не может отличаться от метки больше чем
на единицу, поэтому и допустимый шаг здесь мягче. Но и сходится спуск
медленнее: у log loss нет одного жесткого минимума-чаши, вдали от границы
поверхность почти плоская.

## 7. Вероятности и порог

Логистическая регрессия выдает не ответ, а вероятность. Ответ появляется
после порога: по умолчанию 0.5, и это ровно граница $\langle w, x \rangle + b = 0$.
Но порог — не часть модели, его выбираем мы. Сдвигая его, мы меняем, какие
ошибки модель делает чаще.

Вспомним первое занятие. Precision — доля настоящих объектов сорта 1 среди
тех, кого модель назвала сортом 1. Recall — доля найденных среди всех вин
сорта 1. Смотрим, как они меняются с порогом на тестовой выборке.

In [ ]:
proba_test = sigmoid(X_test @ w_log + b_log)
thresholds = np.linspace(0.05, 0.95, 19)

rows = []
for threshold in thresholds:
    predicted = (proba_test >= threshold).astype(int)
    rows.append({
        "порог": round(threshold, 2),
        "accuracy": accuracy_score(y_test, predicted),
        "precision": precision_score(y_test, predicted, zero_division=0),
        "recall": recall_score(y_test, predicted),
        "названо сортом 1": int(predicted.sum()),
    })
threshold_table = pd.DataFrame(rows).set_index("порог")

fig, ax = plt.subplots(figsize=(8, 4.2))
for column, color in [("accuracy", BLACK), ("precision", BLUE), ("recall", OCHRE)]:
    ax.plot(threshold_table.index, threshold_table[column], "o-", color=color, ms=4, label=column)
ax.axvline(0.5, color=GREY, ls="--", lw=1)
ax.set_xlabel("порог на вероятность")
ax.set_title("Одна модель, разные пороги")
ax.legend()
plt.show()

threshold_table.loc[[0.2, 0.35, 0.5, 0.65, 0.8]].round(3)

Порог — это та же прямая, сдвинутая по плоскости. Вероятность равна $t$ там,
где $\sigma(\langle w, x \rangle + b) = t$, то есть
$\langle w, x \rangle + b = \log\frac{t}{1 - t}$: направление границы то же,
меняется только свободный член. Смотрим на тестовых винах, на которых
считались метрики.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 5.5))
plot_points(ax, X_test, y_test, alpha=0.8)
threshold_colors = [BLUE, "#7FB3E0", BLACK, "#E0A868", OCHRE]
for threshold, color in zip([0.2, 0.35, 0.5, 0.65, 0.8], threshold_colors):
    shift = np.log(threshold / (1 - threshold))
    predicted = (X_test @ w_log + b_log >= shift).astype(int)
    boundary_line(ax, w_log, b_log - shift, color=color,
                  label=f"порог {threshold}: precision {precision_score(y_test, predicted):.2f}, recall {recall_score(y_test, predicted):.2f}")
ax.set_title("Тестовые вина: порог сдвигает границу, не поворачивая ее")
ax.legend(loc="upper right", fontsize=9)
plt.show()

Ниже границы модель отвечает «сорт 1». Низкий порог поднимает границу
вверх и захватывает больше вин, в том числе чужих: recall растет, precision
падает. Высокий порог опускает границу, и часть синих точек остается над ней.

Низкий порог: почти все вина сорта 1 найдены, но вместе с ними много чужих.
Высокий порог: названные сортом 1 почти все настоящие, зато часть сорта 1
пропущена. Модель одна и та же, меняется только цена ошибки, которую мы
выбрали. Какой порог правильный, зависит от задачи, а не от алгоритма;
подробно о метриках и их выборе — на следующем занятии.

## 8. Регуляризация

Переходим ко всем тринадцати признакам. На них сорт 1 отделяется от остальных
идеально: существует прямая, по одну сторону которой все вина сорта 1,
по другую все остальные. Кажется, что это хорошо. Для логистической регрессии
без штрафа это беда, и вот почему.

Запишем log loss через отступы, как в четвертом разделе, с $w$, включающим
свободный член:

$$L(w) = \frac{1}{n}\sum_{i=1}^{n} \log\big(1 + e^{-M_i}\big), \qquad
M_i = y_i^{\pm} \langle w, x_i \rangle$$

Пусть нашлись веса $w$, при которых все отступы положительные: $M_i > 0$
для всех $i$. Умножим веса на два. Отступы линейны по $w$, поэтому они
тоже удвоятся, $M_i \to 2M_i$, и каждое слагаемое уменьшится:

$$\log\big(1 + e^{-2M_i}\big) < \log\big(1 + e^{-M_i}\big),
\qquad\text{так как } e^{-2M_i} < e^{-M_i} \text{ при } M_i > 0$$

Значит $L(2w) < L(w)$, и то же самое для $L(4w)$, $L(8w)$ и дальше.
При $t \to \infty$ все $e^{-tM_i} \to 0$ и $L(tw) \to 0$, но ни при каком
конечном $t$ нуля нет: минимум не достигается, веса растут, пока решатель
не остановится по числу итераций. Вероятности при этом
$\sigma(t\langle w, x_i \rangle) \to 0$ или $1$ для каждого объекта: модель
становится абсолютно уверенной в каждом ответе, включая те, что на тесте
окажутся неверными.

Штраф за большие веса решает и это. В sklearn сила штрафа у логистической
регрессии задается параметром `C`, и он обратный: $C = 1/\lambda$, чем меньше
`C`, тем сильнее штраф. По умолчанию `C = 1` и штраф L2; `C=np.inf` означает
отсутствие штрафа.

In [ ]:
X_all = frame.drop(columns="target").to_numpy()
feature_names = list(frame.drop(columns="target").columns)

X_all_train_raw, X_all_test_raw, y_all_train, y_all_test = train_test_split(
    X_all, y_binary, test_size=0.3, random_state=SEED, stratify=y_binary
)
scaler_all = StandardScaler().fit(X_all_train_raw)
X_all_train = scaler_all.transform(X_all_train_raw)
X_all_test = scaler_all.transform(X_all_test_raw)

no_penalty = LogisticRegression(C=np.inf, max_iter=100000, tol=1e-12).fit(X_all_train, y_all_train)
print(f"без штрафа: самый большой вес по модулю {np.abs(no_penalty.coef_).max():.1f}, "
      f"accuracy на обучении {no_penalty.score(X_all_train, y_all_train):.3f}, на тесте {no_penalty.score(X_all_test, y_all_test):.3f}")
print(f"самое неуверенное предсказание на тесте: {np.abs(no_penalty.predict_proba(X_all_test)[:, 1] - 0.5).min() + 0.5:.3f}, "
      "даже сомнительные вина получают уверенный ответ")

Что именно делает штраф, лучше видно на двух признаках, где вероятность
можно нарисовать цветом. Четыре модели от сильного штрафа до его отсутствия.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 4.8))
for ax, C, penalty_label in zip(axes, [0.01, 0.1, 1, np.inf],
                                ["сильный штраф", "штраф слабее", "штраф еще слабее", "штрафа нет"]):
    model = LogisticRegression(C=C, max_iter=10000).fit(X_train, y_train)
    proba_c = model.predict_proba(grid_points)[:, 1].reshape(grid_x.shape)
    ax.pcolormesh(grid_x, grid_y, proba_c, cmap="RdBu_r", vmin=0, vmax=1, shading="auto", alpha=0.85)
    plot_points(ax, X_train, y_train, alpha=0.6)
    boundary_line(ax, model.coef_[0], model.intercept_[0])
    ax.set_title(f"C = {C}, {penalty_label}\n|w| = {np.linalg.norm(model.coef_):.2f}, log loss на тесте "
                 f"{log_loss(y_test, model.predict_proba(X_test)[:, 1]):.3f}", fontsize=10)
    ax.set_ylabel("")
axes[0].set_ylabel("color_intensity, стандартизованные")
fig.suptitle("Штраф ослабевает слева направо: C растет, λ = 1/C падает, веса растут", fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

Граница почти не двигается, а вот переход от синего к красному меняется
сильно: при сильном штрафе веса маленькие, сигмоида пологая и модель ни в чем
не уверена, без штрафа веса большие и переход резкий. Штраф управляет
не тем, где граница, а тем, насколько модель уверена вдали от нее.

Теперь то же на всех тринадцати признаках. Слева все веса при ослаблении
штрафа, справа распределение вероятностей на тестовых винах.

In [ ]:
C_path = np.logspace(-3, 3, 40)
weight_paths = np.array([
    LogisticRegression(C=C, max_iter=10000).fit(X_all_train, y_all_train).coef_[0] for C in C_path
])

fig, axes = plt.subplots(1, 2, figsize=(15, 4.6), gridspec_kw={"width_ratios": [1.5, 1]})
for j, name in enumerate(feature_names):
    axes[0].plot(C_path, weight_paths[:, j], lw=1.5, label=name)
axes[0].set_xscale("log")
axes[0].axhline(0, color=BLACK, lw=0.8)
axes[0].set_xlabel("C, чем больше, тем слабее штраф")
axes[0].set_ylabel("вес")
axes[0].set_title("Веса растут, пока штраф не остановит")
axes[0].legend(fontsize=7, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)

for C, color in [(0.01, GREY), (1, BLUE), (1000, OCHRE)]:
    model = LogisticRegression(C=C, max_iter=10000).fit(X_all_train, y_all_train)
    axes[1].hist(model.predict_proba(X_all_test)[:, 1], bins=np.linspace(0, 1, 21), alpha=0.6, color=color, label=f"C = {C}")
axes[1].set_xlabel("вероятность сорта 1 на тестовых винах")
axes[1].set_ylabel("сколько вин")
axes[1].set_title("Чем слабее штраф, тем увереннее ответы")
axes[1].legend()
plt.tight_layout()
plt.show()

При `C = 0.01` все вероятности лежат между 0.1 и 0.7: ни одного уверенного
ответа, модель почти ничего не различает. При `C = 1000` они разбежались к нулю и единице, включая вина,
на которых модель ошибается. Где между этими крайностями остановиться,
решает кросс-валидация.

In [ ]:
C_grid = [0.001, 0.01, 0.1, 1, 10, 100, 1000]
cv_binary = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rows = []
for C in C_grid:
    model = LogisticRegression(C=C, max_iter=10000)
    cv_loss = -cross_val_score(model, X_all_train, y_all_train, cv=cv_binary, scoring="neg_log_loss").mean()
    model.fit(X_all_train, y_all_train)
    rows.append({
        "C": C,
        "log loss на кросс-валидации": cv_loss,
        "log loss на тесте": log_loss(y_all_test, model.predict_proba(X_all_test)[:, 1]),
        "accuracy на тесте": model.score(X_all_test, y_all_test),
        "max |w|": np.abs(model.coef_).max(),
    })
C_table = pd.DataFrame(rows).set_index("C")
best_C = C_table["log loss на кросс-валидации"].idxmin()

fig, ax = plt.subplots(figsize=(8, 4.2))
ax.plot(C_grid, C_table["log loss на кросс-валидации"], "o-", color=BLUE, label="кросс-валидация")
ax.plot(C_grid, C_table["log loss на тесте"], "o-", color=OCHRE, label="тест")
ax.set_xscale("log")
ax.set_xlabel("C, чем меньше, тем сильнее штраф")
ax.set_ylabel("log loss")
ax.set_title(f"Лучший C по кросс-валидации: {best_C}")
ax.legend()
plt.show()

C_table.round(3)

Слишком сильный штраф давит веса и модель почти ничего не различает,
слишком слабый возвращает бесконечную уверенность. Обратите внимание:
начиная с `C = 0.1` accuracy на тесте почти не отличает варианты, а log loss
отличает хорошо, потому что видит не только ответ, но и уверенность. Штраф
здесь выбирают не ради доли верных ответов, а ради честных вероятностей.

## 9. Три сорта

Классов три, а сигмоида умеет только два. Два способа обобщить.

**Один против всех.** Обучаем три бинарных классификатора: сорт 0 против
остальных, сорт 1 против остальных, сорт 2 против остальных. Отвечаем тем
классом, чей классификатор увереннее.

**Softmax.** Обобщение сигмоиды на $K$ классов. Откуда оно берется, видно,
если переписать сигмоиду. Она говорит, что логарифм отношения шансов
линеен по признакам:

$$\log\frac{p(y = 1 \mid x)}{p(y = 0 \mid x)} = \langle w, x \rangle + b$$

Проверьте: если $p = \sigma(z)$, то $1 - p = \frac{e^{-z}}{1 + e^{-z}}$
и $\frac{p}{1 - p} = e^{z}$. Ту же сигмоиду можно записать и как долю
одной экспоненты в сумме двух, если классу 0 дать очко $0$, а классу 1 —
очко $z$:

$$\sigma(z) = \frac{1}{1 + e^{-z}} = \frac{e^{z}}{e^{z} + e^{0}}$$

Для $K$ классов делаем то же самое. Каждому классу даем свои веса
и свое линейное очко $z_k = \langle w_k, x \rangle$ и требуем, чтобы логарифм
отношения вероятностей любых двух классов был разностью очков:

$$\log\frac{p(y = k \mid x)}{p(y = j \mid x)} = z_k - z_j
\quad\Longrightarrow\quad p(y = k \mid x) = p(y = j \mid x)\, e^{z_k - z_j}$$

Значит, все вероятности пропорциональны $e^{z_k}$: $p(y = k \mid x) = c\, e^{z_k}$
с одной общей константой $c$. Она находится из того, что вероятности
суммируются в единицу: $c \sum_j e^{z_j} = 1$. Отсюда

$$p(y = k \mid x) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}
= \frac{e^{\langle w_k, x \rangle}}{\sum_j e^{\langle w_j, x \rangle}}$$

При $K = 2$ и очках $(0, z)$ это ровно сигмоида. Функция потерь — та же
кросс-энтропия из пятого раздела в индикаторной записи, минус средний
логарифм вероятности правильного класса:

$$L = -\frac{1}{n}\sum_{i=1}^{n} \log p(y_i \mid x_i)
= -\frac{1}{n}\sum_{i=1}^{n} \sum_{k=1}^{K} [y_i = k]\, \log p(y = k \mid x_i)$$

Это максимум правдоподобия, только вместо распределения Бернулли стоит
категориальное: $K$ исходов вместо двух. Именно этот вариант стоит в sklearn
по умолчанию для многоклассовой задачи.

Напишем softmax. Технический момент: экспонента от большого числа
переполняется, уже $e^{1000}$ не помещается в `float64`. Поэтому перед
экспонентой из всех очков объекта вычитают их максимум $m = \max_j z_j$,
где $z_j = \langle w_j, x \rangle$. Вероятности от этого не меняются:
$e^{z_k - m} = e^{z_k} \cdot e^{-m}$, и множитель $e^{-m}$ появляется
и в числителе, и в каждом слагаемом знаменателя:

$$\frac{e^{z_k - m}}{\sum_j e^{z_j - m}}
= \frac{e^{z_k}\, e^{-m}}{e^{-m} \sum_j e^{z_j}}
= \frac{e^{z_k}}{\sum_j e^{z_j}}$$

Зато после вычитания самое большое очко равно нулю, все экспоненты
не больше единицы, и переполнения нет.

In [ ]:
# scores: (n, k) — по числу на каждый класс для каждого объекта
# возвращает вероятности: (n, k), в каждой строке сумма равна единице
def softmax(scores: np.ndarray) -> np.ndarray:
    return ...

In [ ]:
# --- проверка ---

# 1. строки суммируются в единицу, все вероятности в (0, 1)
probs = softmax(np.array([[1.0, 2.0, 3.0], [0.0, 0.0, 0.0]]))
assert np.allclose(probs.sum(axis=1), 1), "строки softmax должны суммироваться в единицу"
assert np.allclose(probs[1], [1 / 3, 1 / 3, 1 / 3]), "при равных очках вероятности равны"

# 2. сдвиг всех очков на константу ничего не меняет
assert np.allclose(softmax(np.array([[1.0, 2.0]])), softmax(np.array([[101.0, 102.0]]))), "softmax не должен зависеть от сдвига"

# 3. большие числа не ломают
assert np.isfinite(softmax(np.array([[1000.0, 1000.0, 0.0]]))).all(), "переполнение: вычитайте максимум строки"

# 4. для двух классов это сигмоида разности
z = np.array([[0.3, -1.2]])
assert np.isclose(softmax(z)[0, 0], sigmoid(z[0, 0] - z[0, 1])), "для двух классов softmax должен совпадать с сигмоидой"

print("проверки пройдены")

In [ ]:
y_multi = frame["target"].to_numpy()
X_m_train_raw, X_m_test_raw, y_m_train, y_m_test = train_test_split(
    X_all, y_multi, test_size=0.3, random_state=SEED, stratify=y_multi
)
scaler_multi = StandardScaler().fit(X_m_train_raw)
X_m_train, X_m_test = scaler_multi.transform(X_m_train_raw), scaler_multi.transform(X_m_test_raw)

multi_model = LogisticRegression(max_iter=10000).fit(X_m_train, y_m_train)

# вероятности sklearn это ровно softmax от <w_k, x> + b_k
scores_test = X_m_test @ multi_model.coef_.T + multi_model.intercept_
assert np.allclose(softmax(scores_test), multi_model.predict_proba(X_m_test))

print(f"accuracy на тесте: {multi_model.score(X_m_test, y_m_test):.3f}")
print("матрица ошибок, строки — настоящий сорт, столбцы — предсказанный:")
print(confusion_matrix(y_m_test, multi_model.predict(X_m_test)))
print("\nсамое сомнительное тестовое вино, вероятности сортов:",
      np.round(multi_model.predict_proba(X_m_test)[multi_model.predict_proba(X_m_test).max(axis=1).argmin()], 3))

In [ ]:
# какие признаки тянут к какому сорту: веса softmax после стандартизации
weights_multi = pd.DataFrame(multi_model.coef_.T, index=feature_names, columns=["сорт 0", "сорт 1", "сорт 2"])
weights_multi.round(2).style.background_gradient(cmap="RdBu", vmin=-1.5, vmax=1.5)

Каждый столбец читается как в регрессии: положительный вес — признак тянет
к этому сорту, отрицательный — от него. Пролин и интенсивность цвета тянут
к сорту 0, низкая интенсивность цвета — к сорту 1, низкие флавоноиды —
к сорту 2. Это те же признаки, по которым сорта различались на глаз
на первом занятии.

## 10. Метод опорных векторов

### Какую прямую выбрать

Вернемся к двум признакам и начнем с простого случая: классы разделимы.
Прямых без единой ошибки бесконечно много, и логистическая регрессия без
штрафа среди них выбрать не может: любую из них можно сделать сколь угодно
уверенной, увеличивая веса. Нужен другой критерий.

Возьмем два облака и проведем три прямые, ни одна не ошибается. Чем они
отличаются? Расстоянием до ближайшего объекта. Прямая, которая проходит
вплотную к чьей-то точке, при малейшем сдвиге данных начнет ошибаться.
Прямая, от которой до обоих классов далеко, устойчивее. Так что выбираем
прямую с **самым широким зазором** до ближайших объектов.

In [ ]:
rng_svm = np.random.default_rng(7)
cloud_a = rng_svm.normal([0, 0], 0.7, (40, 2))
cloud_b = rng_svm.normal([3, 3], 0.7, (40, 2))
X_sep = np.vstack([cloud_a, cloud_b])
y_sep = np.r_[np.zeros(40, dtype=int), np.ones(40, dtype=int)]

# три прямые через середину между облаками с разным наклоном
midpoint = np.array([1.5, 1.5])
candidates = {
    "прямая 1": np.array([1.0, 1.0]),
    "прямая 2": np.array([0.25, 1.0]),
    "прямая 3": np.array([1.0, 0.25]),
}
svm_sep = LinearSVC(C=100, max_iter=100000).fit(X_sep, y_sep)


def gap_to_nearest(w, b):
    return (np.abs(X_sep @ w + b) / np.linalg.norm(w)).min()


fig, ax = plt.subplots(figsize=(6.5, 6))
ax.scatter(cloud_a[:, 0], cloud_a[:, 1], s=22, color=GREY, label="класс 0")
ax.scatter(cloud_b[:, 0], cloud_b[:, 1], s=22, color=BLUE, label="класс 1")
xs = np.linspace(-2, 5, 100)
for (name, w), color in zip(candidates.items(), [GREY, "#7A63B8", "#C7B45A"]):
    b = -w @ midpoint
    errors = int((margins_of(X_sep, y_sep, w, b) < 0).sum())
    ax.plot(xs, -(w[0] * xs + b) / w[1], color=color, lw=1.6, label=f"{name}: зазор {gap_to_nearest(w, b):.2f}, ошибок {errors}")
w_sep, b_sep = svm_sep.coef_[0], svm_sep.intercept_[0]
ax.plot(xs, -(w_sep[0] * xs + b_sep) / w_sep[1], color=OCHRE, lw=2.5, label=f"SVM: зазор {gap_to_nearest(w_sep, b_sep):.2f}")
for shift in (-1, 1):
    ax.plot(xs, -(w_sep[0] * xs + b_sep - shift) / w_sep[1], color=OCHRE, lw=1, ls="--")
support_sep = margins_of(X_sep, y_sep, w_sep, b_sep) <= 1 + 1e-6
ax.scatter(X_sep[support_sep, 0], X_sep[support_sep, 1], s=140, facecolor="none", edgecolor=BLACK, lw=1.3, label="опорные векторы")
ax.set_xlim(-2, 5)
ax.set_ylim(-2, 5)
ax.set_aspect("equal")
ax.set_title("Все прямые без ошибок, но зазор у них разный")
ax.legend(loc="upper left", fontsize=9)
plt.show()

SVM нашел прямую с самым широким зазором, и на нее опираются всего несколько
объектов, обведенных черным: те, что лежат ровно на краю полосы. Остальные
на положение границы не влияют вообще. Отсюда название: **опорные векторы**.

### Как это записать

Сначала разберемся, как измерить ширину полосы. Пара $(w, b)$ задает
границу с точностью до множителя: прямые $\langle w, x \rangle + b = 0$
и $\langle 2w, x \rangle + 2b = 0$ — это одна и та же прямая. Множитель
фиксируем так, чтобы ближайшие к границе объекты имели отступ ровно 1:
тогда края полосы — это прямые $\langle w, x \rangle + b = \pm 1$.

**Расстояние от точки до прямой.** Вектор $w$ перпендикулярен границе.
Возьмем точку $x_1$ на краю $\langle w, x \rangle + b = 1$ и сдвинем ее вдоль $w$
на величину $d$, то есть в точку $x_1 - d \frac{w}{\|w\|}$. Подставим
в линейную функцию:

$$\Big\langle w,\ x_1 - d\frac{w}{\|w\|} \Big\rangle + b
= \underbrace{\langle w, x_1 \rangle + b}_{= 1} - d\,\frac{\langle w, w \rangle}{\|w\|}
= 1 - d\,\|w\|$$

На границе это выражение равно нулю, значит до границы $d = 1 / \|w\|$.
На противоположном краю оно равно $-1$, значит до него $d = 2 / \|w\|$.
Это и есть ширина полосы:

$$\text{ширина} = \frac{2}{\|w\|}$$

Чем меньше норма весов, тем шире полоса. Требование «все объекты вне
полосы и на своей стороне» — это $M_i \geq 1$ для всех $i$. Значит, самая
широкая полоса — это наименьшая норма $\|w\|$ при условии $M_i \geq 1$.

Когда классы не разделимы, условие выполнить нельзя. Разрешаем его нарушать,
но за каждое нарушение платим ровно на столько, на сколько отступ не дотянул
до единицы: $\max(0, 1 - M_i)$. Это оранжевая кривая с графика потерь, hinge.
Складываем со штрафом за норму:

$$\frac{1}{n}\sum_{i=1}^{n}\max(0, 1 - M_i) + \frac{1}{2C}\|w\|^2 \to \min_{w, b}$$

`C` решает, что важнее: широкая полоса или мало нарушений. Опорными векторами
теперь называются все объекты с $M_i \leq 1$: на краю полосы, внутри нее
и на чужой стороне. Только они входят в функцию потерь с ненулевым вкладом,
только они и определяют границу.

### На вине

Третья модель проходит через те же две функции, что и первые две.

In [ ]:
svm_model = LinearSVC(C=1, max_iter=100000).fit(X_train, y_train)
w_svm, b_svm = svm_model.coef_[0], svm_model.intercept_[0]
support = margins_of(X_train, y_train, w_svm, b_svm) <= 1 + 1e-6

show_model("SVM, C=1", w_svm, b_svm, band=True, support=support)
print(f"опорных векторов: {support.sum()} из {len(y_train)}, ширина полосы {2 / np.linalg.norm(w_svm):.2f}")
evaluate_model("SVM, C=1", w_svm, b_svm)

Классы пересекаются, поэтому полоса широкая и опорных векторов много: все,
кто попал внутрь или на чужую сторону. Отступы SVM устроены иначе, чем
у логистической регрессии: у нее они разбегаются далеко вправо, у SVM нет
никакого стимула делать отступ больше единицы, и гистограмма прижата к нулю.

Все три границы на одной картинке.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5.5))
plot_points(ax, X_train, y_train, alpha=0.35)
boundary_line(ax, w_reg, b_reg, color=GREY, label="линейная регрессия на метках")
boundary_line(ax, w_log, b_log, color=BLUE, label="логистическая регрессия")
boundary_line(ax, w_svm, b_svm, color=OCHRE, label="SVM, C=1", band=True)
ax.set_title("Три модели, одна плоскость")
ax.legend(loc="upper right")
plt.show()

pd.DataFrame(results).T.round(3)

Логистическая регрессия и SVM провели почти одну прямую и ошибаются на одних
и тех же винах: обе функции потерь не трогают уверенно правильные ответы,
и далекие вина других сортов их не сбивают. Линейная регрессия на метках стоит
особняком именно из-за этих вин. Метрики на тесте у логистической регрессии
и SVM отличаются на одно вино из 54, это в пределах шума. Разница между
ними не в этой прямой, а в том, что они считают уверенностью, как ведут себя
на разделимых данных и от чего зависит их граница.

### Что делает `C`

Смотрим границу и полосу при четырех значениях `C` в одинаковых осях.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9.5))
for ax, C in zip(axes.flat, [0.01, 0.1, 1, 100]):
    model = LinearSVC(C=C, max_iter=100000).fit(X_train, y_train)
    w_c, b_c = model.coef_[0], model.intercept_[0]
    support_c = margins_of(X_train, y_train, w_c, b_c) <= 1 + 1e-6
    plot_points(ax, X_train, y_train, alpha=0.35)
    boundary_line(ax, w_c, b_c, color=OCHRE, band=True)
    ax.scatter(X_train[support_c, 0], X_train[support_c, 1], s=110, facecolor="none", edgecolor=BLACK, lw=1.0)
    ax.set_title(f"C = {C}: ширина полосы {2 / np.linalg.norm(w_c):.2f}, опорных {support_c.sum()}, "
                 f"accuracy на тесте {model.score(X_test, y_test):.3f}", fontsize=11)
plt.tight_layout()
plt.show()

При маленьком `C` штраф за веса важнее ошибок: полоса широкая, почти все вина
внутри, все они опорные. При большом `C` важнее ошибки: полоса сужается,
опорных остается меньше, граница подстраивается под отдельные объекты.
Выбирается `C`, как и все гиперпараметры, по кросс-валидации.

### Чем определяется граница

Проверим утверждение про опорные векторы прямо. Уберем из обучающей выборки
все вина, которые не опорные, и обучим SVM заново. Потом вернем их и уберем
один опорный вектор.

In [ ]:
def refit_svm(mask):
    model = LinearSVC(C=1, max_iter=100000).fit(X_train[mask], y_train[mask])
    return model.coef_[0], model.intercept_[0]


w_only_support, b_only_support = refit_svm(support)
most_important = np.argmin(margins_of(X_train, y_train, w_svm, b_svm))    # опорный вектор с самым маленьким отступом
without_one = np.ones(len(y_train), dtype=bool)
without_one[most_important] = False
w_without_one, b_without_one = refit_svm(without_one)

print("исходная граница:                          w =", np.round(w_svm, 3), " b =", round(b_svm, 3))
print(f"только {support.sum()} опорных векторов:           w =", np.round(w_only_support, 3), " b =", round(b_only_support, 3))
print("без одного опорного вектора:                w =", np.round(w_without_one, 3), " b =", round(b_without_one, 3))

fig, ax = plt.subplots(figsize=(7, 5.5))
plot_points(ax, X_train, y_train, alpha=0.3)
boundary_line(ax, w_svm, b_svm, color=OCHRE, label="все вина")
boundary_line(ax, w_only_support, b_only_support, color=BLACK, label="только опорные векторы")
boundary_line(ax, w_without_one, b_without_one, color=BLUE, label="без одного опорного вектора")
ax.scatter(X_train[most_important, 0], X_train[most_important, 1], s=200, facecolor="none", edgecolor=BLUE, lw=2, label="убранный вектор")
ax.set_title("От каких объектов зависит граница")
ax.legend(loc="upper right")
plt.show()

Выбросили сорок процентов выборки, а граница не сдвинулась ни на тысячную:
объекты вдали от полосы ничего в SVM не решают. Убрали один объект с самым
отрицательным отступом, то самое вино сорта 1 далеко справа, — граница
повернулась. Логистическая регрессия устроена иначе: у нее вклад каждого
объекта убывает с отступом, но никогда не становится нулевым, и любое вино
хоть немного, но двигает ее границу.

## 11. Что запомнить

**Классификация — та же линейная модель.** Меняется только то, как выход
превращается в ответ: знак, порог на вероятность или класс с наибольшим
softmax.

**Отступ** $M = y(\langle w, x \rangle + b)$ — одна величина, через которую
записываются все функции потерь: квадратичная, логистическая и hinge.
Квадратичная штрафует уверенно правильные ответы, поэтому линейная регрессия на метках
ломается на далеких объектах.

**Логистическая регрессия** — максимум правдоподобия при ответе из распределения
Бернулли, ровно как метод наименьших квадратов при нормальном шуме. Формулы
для весов нет, работает градиентный спуск.

**Порог** — не часть модели. Он меняет соотношение precision и recall,
и его выбирает тот, кто знает цену ошибки.

**Регуляризация** нужна и здесь: на разделимых данных без штрафа веса растут
бесконечно. `C` в sklearn обратен силе штрафа.

**SVM** — самая широкая полоса между классами; за объекты внутри полосы
и на чужой стороне платит hinge loss, и только эти объекты, опорные векторы,
определяют границу.

**Сравнивать модели** имеет смысл только в одном сетапе: одни данные, одни
метрики, одна картинка. На наших винах логистическая регрессия и SVM проводят
почти одну прямую, а линейную регрессию на метках сбивают далекие объекты.

| ответ | предположение | функция потерь | метод |
|---|---|---|---|
| число | нормальный шум | сумма квадратов | линейная регрессия |
| 0 или 1 | распределение Бернулли | log loss | логистическая регрессия |
| −1 или +1 | нужна широкая полоса | hinge | SVM |
| один из k классов | категориальное распределение | кросс-энтропия | softmax-регрессия |

## К следующему занятию

1. Посмотреть лекцию [Cross validation. BVD](https://www.youtube.com/watch?v=-taRv4WTUbo&list=PL4_hYwCyhAvZyW6qS58x4uElZgAkMVUvj) Лектория ФПМИ
2. Сделать лабораторную: она домашняя, прислать преподавателю в личные сообщения до 26 сентября
3. На следующем занятии выдается первая домашняя работа: нужен аккаунт на [huggingface.co](https://huggingface.co)